# De artículos científicos a una tabla con un LLM local

Partiendo de una muestra de artículos científicos de acceso abierto (revista *Frontiers*, sobre placebo, dolor y depresión), el objetivo es construir una tabla con **una fila por artículo** y **una columna por variable de interés**. Si una variable no aparece en un artículo, se deja como nulo.

Para leer los artículos uso un **LLM de Hugging Face ejecutado en local** (`Qwen3-4B-Instruct-2507`), sin APIs de pago: los documentos no salen del entorno de ejecución.

## Cómo funciona

1. **Descarga.** Cada artículo se descarga a partir de su DOI desde Frontiers; si falla, se usa el texto completo de la API de Europe PMC.
2. **Texto.** PyMuPDF extrae el texto del PDF. Se quita la bibliografía y se limita la longitud para que quepa en el contexto del modelo.
3. **Extracción.** El LLM devuelve un JSON con tres variables; si una no aparece, devuelve `null`.
4. **Tabla.** Cada respuesta se añade como una fila (`append`) y con todas se construye un `DataFrame`, que se exporta a CSV con los nulos como `NULL`.

| Columna | Qué recoge |
|---|---|
| `condicion` | Enfermedad, síntoma o condición en la que se centra el estudio |
| `n_participantes` | Número total de personas reclutadas (nulo en revisiones, artículos teóricos o datos de redes sociales) |
| `intervencion` | Fármaco, placebo o tratamiento aplicado o evaluado |

> En Colab conviene activar la GPU (T4). Sin GPU se usa automáticamente un modelo más pequeño, `Qwen2.5-1.5B-Instruct`.

In [30]:
# Instalamos las librerías necesarias:
#  - pymupdf: para extraer el texto de los PDFs
#  - requests: para descargar los artículos de internet
#  - transformers + accelerate: para descargar y ejecutar el LLM de Hugging Face
!pip install -q pymupdf requests transformers accelerate

In [31]:
import os
import re
import json
import ast

import fitz            # PyMuPDF
import requests
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

In [32]:
# ---------------------------------------------------------------
# Documentos a analizar: nombre del fichero -> DOI del artículo
# (son artículos open access de la revista Frontiers)
# ---------------------------------------------------------------
REFERENCIAS = {
    'fpsyg-08-00308.pdf':   '10.3389/fpsyg.2017.00308',
    'fpsyg-09-01240.pdf':   '10.3389/fpsyg.2018.01240',
    'fpsyg-11-01354.pdf':   '10.3389/fpsyg.2020.01354',
    'fpsyg-12-639236.pdf':  '10.3389/fpsyg.2021.639236',
    'fpsyt-14-1301143.pdf': '10.3389/fpsyt.2023.1301143',
    'fpubh-10-1045777.pdf': '10.3389/fpubh.2022.1045777',
}

CARPETA_PDFS = 'pdfs'                       # aquí se guardan los PDFs descargados
os.makedirs(CARPETA_PDFS, exist_ok=True)

# Cabecera de navegador: algunas webs rechazan peticiones sin User-Agent
HEADERS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                         '(KHTML, like Gecko) Chrome/124.0 Safari/537.36'}


def descargar_pdf(doi, ruta_destino):
    """Descarga el PDF del artículo desde Frontiers a partir de su DOI.
    Devuelve True si se ha descargado un PDF válido."""
    if os.path.exists(ruta_destino):        # si ya lo tenemos, no lo volvemos a descargar
        return True
    url = f'https://www.frontiersin.org/articles/{doi}/pdf'
    try:
        r = requests.get(url, headers=HEADERS, timeout=60, allow_redirects=True)
        if r.status_code == 200 and r.content[:4] == b'%PDF':   # comprobamos que es un PDF de verdad
            with open(ruta_destino, 'wb') as f:
                f.write(r.content)
            return True
    except requests.RequestException as e:
        print(f'  No se pudo descargar el PDF: {e}')
    return False


def texto_europepmc(doi):
    """Respaldo: obtiene el texto completo del artículo desde la API pública de Europe PMC."""
    try:
        busqueda = requests.get('https://www.ebi.ac.uk/europepmc/webservices/rest/search',
                                params={'query': f'DOI:"{doi}"', 'format': 'json', 'resultType': 'lite'},
                                timeout=60).json()
        resultados = busqueda['resultList']['result']
        if not resultados or 'pmcid' not in resultados[0]:
            return None
        pmcid = resultados[0]['pmcid']
        xml = requests.get(f'https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/fullTextXML',
                           timeout=60).text
        texto = re.sub(r'<[^>]+>', ' ', xml)                 # quitamos las etiquetas XML
        return re.sub(r'\s+', ' ', texto).strip()
    except Exception as e:
        print(f'  Europe PMC tampoco ha funcionado: {e}')
        return None


# Descargamos todos los PDFs y comprobamos qué tenemos
for nombre, doi in REFERENCIAS.items():
    ok = descargar_pdf(doi, os.path.join(CARPETA_PDFS, nombre))
    print(f'{nombre:24s} {"descargado" if ok else "NO descargado (se usará Europe PMC)"}')

fpsyg-08-00308.pdf       descargado
fpsyg-09-01240.pdf       descargado
fpsyg-11-01354.pdf       descargado
fpsyg-12-639236.pdf      descargado
fpsyt-14-1301143.pdf     descargado
fpubh-10-1045777.pdf     descargado


In [33]:
# LLM de Hugging Face. Elegimos el tamaño según el hardware disponible:
#  - Con GPU: Qwen3-4B-Instruct-2507 (modelo reciente, muy bueno siguiendo instrucciones y devolviendo JSON)
#  - Sin GPU (CPU): Qwen2.5-1.5B-Instruct (más ligero, para que no tarde demasiado)
GPU = torch.cuda.is_available()
MODELO = 'Qwen/Qwen3-4B-Instruct-2507' if GPU else 'Qwen/Qwen2.5-1.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODELO)
model = AutoModelForCausalLM.from_pretrained(
    MODELO,
    torch_dtype=torch.float16 if GPU else torch.bfloat16,  # float16 en la GPU T4; bfloat16 en CPU para ahorrar memoria
    device_map='auto'          # usa la GPU automáticamente si existe
)
model.eval()
print(f'Modelo {MODELO} cargado en: {model.device}')

Modelo Qwen/Qwen3-4B-Instruct-2507 cargado en: cuda:0


In [34]:
# ---------------------------------------------------------------
# Variables de interés: nombre de columna -> instrucción para el LLM
# ---------------------------------------------------------------
VARIABLES = {
    'condicion':       'the disease, symptom or health condition the paper focuses on, in 1-3 words '
                       '(e.g. "asthma", "chronic pain", "anxiety"). null if there is none.',
    'n_participantes': 'the TOTAL number of human participants recruited in THIS study, as an integer '
                       '(if there are several experiments, add them up). '
                       'null if the paper is a review, a theoretical/conceptual or methods paper, '
                       'or only analyses existing datasets (e.g. social media posts).',
    'intervencion':    'the drug, placebo or treatment that is administered to participants or evaluated, in 1-4 words '
                       '(e.g. "ibuprofen", "acupuncture", "placebo pill"). null if there is none.',
}

# Número máximo de caracteres del artículo que le pasamos al modelo (~5.000 tokens)
MAX_CHARS = 20000

# Aquí guardamos la respuesta "en bruto" del modelo para cada documento (útil para revisar)
RESPUESTAS = {}


def extraer_texto_pdf(ruta_pdf):
    """Extrae y limpia todo el texto de un PDF (scraping del documento)."""
    with fitz.open(ruta_pdf) as doc:
        texto = ' '.join(pagina.get_text() for pagina in doc)
    return re.sub(r'\s+', ' ', texto).strip()      # quitamos saltos de línea y espacios repetidos


def preparar_texto(texto):
    """Quita la bibliografía (no aporta nada y confunde al modelo) y recorta el texto."""
    corte = re.search(r'\bREFERENCES\b|\bReferences\b', texto[len(texto)//2:])   # buscamos en la 2ª mitad
    if corte:
        texto = texto[:len(texto)//2 + corte.start()]
    return texto[:MAX_CHARS]


def construir_prompt(texto):
    """Crea los mensajes para el LLM con una plantilla JSON exacta."""
    reglas = '\n'.join(f'- "{k}": {v}' for k, v in VARIABLES.items())
    plantilla = json.dumps({k: '...' for k in VARIABLES})
    sistema = ('You are an expert assistant that extracts structured data from scientific papers. '
               'You answer ONLY with a valid JSON object, with no explanations and no markdown.')
    usuario = (f'Read the paper below and extract these variables:\n{reglas}\n\n'
               f'Answer with exactly this JSON structure and these exact keys:\n{plantilla}\n'
               'Use null (without quotes) when the information does not appear in the paper. Do not invent data.\n\n'
               f'PAPER:\n"""{preparar_texto(texto)}"""')
    return [{'role': 'system', 'content': sistema},
            {'role': 'user', 'content': usuario}]


def consultar_llm(mensajes, max_new_tokens=150):
    """Envía los mensajes al LLM local y devuelve su respuesta en texto."""
    # return_dict=True -> obtenemos un diccionario con input_ids y attention_mask
    entrada = tokenizer.apply_chat_template(mensajes, add_generation_prompt=True,
                                            return_tensors='pt', return_dict=True).to(model.device)
    with torch.no_grad():
        salida = model.generate(**entrada, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
    n_prompt = entrada['input_ids'].shape[-1]          # longitud del prompt
    # Nos quedamos solo con los tokens nuevos (la respuesta), no con el prompt
    return tokenizer.decode(salida[0][n_prompt:], skip_special_tokens=True)


VALORES_NULOS = ('', 'null', 'none', 'n/a', 'na', 'nan', 'not reported', 'not mentioned',
                 'not applicable', 'unknown', '...')

def parsear_respuesta(respuesta):
    """Convierte la respuesta del LLM en un diccionario con las variables.
    Si algo falla o falta, imputa un nulo (None)."""
    fila = {k: None for k in VARIABLES}                       # por defecto todo nulo
    limpio = re.sub(r'```(?:json)?', '', respuesta)           # quitamos posibles bloques ```json
    encontrado = re.search(r'\{.*\}', limpio, re.DOTALL)
    if not encontrado:
        return fila
    try:
        datos = json.loads(encontrado.group())
    except json.JSONDecodeError:
        try:                                                  # 2º intento: JSON con comillas simples
            datos = ast.literal_eval(encontrado.group().replace('null', 'None'))
        except Exception:
            return fila

    datos = {str(k).strip().lower(): v for k, v in datos.items()}   # claves sin mayúsculas/espacios
    for k in VARIABLES:
        valor = datos.get(k)
        if valor is None or str(valor).strip().lower() in VALORES_NULOS:
            valor = None
        fila[k] = valor

    # n_participantes debe ser un número entero
    if fila['n_participantes'] is not None:
        numeros = re.findall(r'\d+', str(fila['n_participantes']).replace(',', ''))
        fila['n_participantes'] = int(numeros[0]) if numeros else None
        if fila['n_participantes'] == 0:                      # 0 participantes = no hay participantes
            fila['n_participantes'] = None
    return fila


def obtener_texto(nombre, doi):
    """Devuelve el texto de un artículo: del PDF descargado si existe,
    y si no, del respaldo de Europe PMC."""
    ruta = os.path.join(CARPETA_PDFS, nombre)
    if os.path.exists(ruta):
        return extraer_texto_pdf(ruta)
    return texto_europepmc(doi)


def tabla_desde_pdfs(referencias):
    """Recorre todos los documentos, extrae las variables con el LLM y devuelve
    un DataFrame (una fila por documento, una columna por variable)."""
    filas = []                                          # aquí vamos haciendo el append
    for nombre, doi in referencias.items():
        print(f'Procesando {nombre}...')
        try:
            texto = obtener_texto(nombre, doi)
            if not texto:                               # no hemos podido conseguir el documento
                raise ValueError('no se ha podido obtener el texto')
            respuesta = consultar_llm(construir_prompt(texto))
            RESPUESTAS[nombre] = respuesta
            print('   respuesta del modelo:', respuesta.strip().replace('\n', ' '))
            fila = parsear_respuesta(respuesta)
        except Exception as e:                          # si un documento falla, no paramos todo
            print(f'  Error en {nombre}: {type(e).__name__}: {e}')
            fila = {k: None for k in VARIABLES}         # imputamos nulos en toda la fila
        fila['documento'] = nombre
        filas.append(fila)                              # append de la fila del documento

    tabla = pd.DataFrame(filas).set_index('documento')[list(VARIABLES)]
    # Unificamos los nulos: pandas mostraría NaN en columnas numéricas y None en las de texto
    tabla['n_participantes'] = tabla['n_participantes'].astype('Int64')
    tabla = tabla.astype(object).where(tabla.notna(), None)
    return tabla

In [35]:
# Ejecutamos todo el proceso
tabla = tabla_desde_pdfs(REFERENCIAS)
tabla

Procesando fpsyg-08-00308.pdf...
   respuesta del modelo: {"condicion": "acute pain", "n_participantes": 46, "intervencion": "placebo and nocebo"}
Procesando fpsyg-09-01240.pdf...
   respuesta del modelo: {"condicion": "depression", "n_participantes": null, "intervencion": null}
Procesando fpsyg-11-01354.pdf...
   respuesta del modelo: {"condicion": "anxiety", "n_participantes": null, "intervencion": null}
Procesando fpsyg-12-639236.pdf...
   respuesta del modelo: {"condicion": "stress", "n_participantes": 152, "intervencion": "placebo cream"}
Procesando fpsyt-14-1301143.pdf...
   respuesta del modelo: {"condicion": "depression", "n_participantes": null, "intervencion": null}
Procesando fpubh-10-1045777.pdf...
   respuesta del modelo: {"condicion": "depression", "n_participantes": null, "intervencion": null}


,condicion,n_participantes,intervencion
documento,,,
fpsyg-08-00308.pdf,acute pain,46,placebo and nocebo
fpsyg-09-01240.pdf,depression,None,None
fpsyg-11-01354.pdf,anxiety,None,None
fpsyg-12-639236.pdf,stress,152,placebo cream
fpsyt-14-1301143.pdf,depression,None,None
fpubh-10-1045777.pdf,depression,None,None


In [36]:
# Comprobamos cuántos nulos se han imputado en cada variable
print(tabla.isnull().sum())

# Guardamos la tabla en CSV, escribiendo los nulos como 'NULL' para que se vean en Excel
tabla.to_csv('tabla_articulos.csv', na_rep='NULL')

# Tabla final con los nulos escritos como NULL
tabla.fillna('NULL')

condicion          0
n_participantes    4
intervencion       4
dtype: int64


,condicion,n_participantes,intervencion
documento,,,
fpsyg-08-00308.pdf,acute pain,46,placebo and nocebo
fpsyg-09-01240.pdf,depression,NULL,NULL
fpsyg-11-01354.pdf,anxiety,NULL,NULL
fpsyg-12-639236.pdf,stress,152,placebo cream
fpsyt-14-1301143.pdf,depression,NULL,NULL
fpubh-10-1045777.pdf,depression,NULL,NULL


## Problemas que encontré

- **El notebook solo funcionaba con mis archivos.** Al principio leía los PDFs de una carpeta de mi Google Drive, así que a nadie más le funcionaba. Lo cambié para descargar cada artículo por su DOI, con Europe PMC como respaldo.
- **Todas las variables salían `None`.** La tabla se generaba vacía porque, en la versión actual de `transformers`, `tokenizer.apply_chat_template(..., return_tensors="pt")` ya no devuelve un tensor, sino un diccionario. Se arregla con `return_dict=True` y `model.generate(**entrada)`.
- **Una variable que el modelo no sabía rellenar.** Al principio una de las columnas era el tipo de estudio (ensayo, revisión, artículo teórico…), pero el modelo la clasificaba mal una y otra vez. La sustituí por `condicion`, que aparece explícita en el texto, y pasé de `Qwen2.5-3B-Instruct` a `Qwen3-4B-Instruct-2507`.
- **Los nulos no se veían en el CSV.** pandas los escribe como celdas vacías; con `na_rep='NULL'` quedan visibles.

## Limitaciones

La calidad depende del modelo, así que conviene revisar sus respuestas contra los artículos (en el notebook se guarda la respuesta en bruto de cada uno). Solo se le pasan los primeros 20 000 caracteres de cada artículo. Con más documentos, lo siguiente sería medir el porcentaje de aciertos contra una pequeña tabla anotada a mano.